# Stratified 5-way split for `test.csv`

This notebook splits `test.csv` into 5 near-equal parts while preserving the `label` distribution as closely as possible. If the row count is not divisible by 5, the remainder is distributed across the folds automatically.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_DIR = Path.cwd()
INPUT_PATH = DATA_DIR / "test.csv"
OUTPUT_TEMPLATE = "test_part_{part}.csv"
STRATIFY_COL = "label"
N_SPLITS = 5
RANDOM_STATE = 42

In [3]:
df = pd.read_csv(INPUT_PATH)

print(f"Loaded {len(df)} rows from {INPUT_PATH.name}")
print(f"Columns: {list(df.columns)}")
df.head()

Loaded 247 rows from test.csv
Columns: ['text', 'cEXT', 'cNEU', 'cAGR', 'cCON', 'cOPN', 'label']


,text,cEXT,cNEU,cAGR,cCON,cOPN,label
0,It is now 12:32 and so I cannot wait until 12:...,high,high,low,high,high,11011
1,Today has seriously been the longest day ever ...,low,high,low,high,low,1010
2,Well currently I am stressed out. It is as if ...,high,high,low,high,high,11011
3,it is wednesday. I can't wait until friday bec...,low,high,high,low,high,10110
4,I feel nervous inside and butterflies keep fly...,high,high,high,high,low,1111


In [4]:
# Shuffle once for reproducibility, then split each label group across the 5 folds.
# Larger chunks are assigned to the currently smallest fold to keep total fold sizes balanced.
shuffled_df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

fold_indices = [[] for _ in range(N_SPLITS)]

for _, group in shuffled_df.groupby(STRATIFY_COL, sort=False):
    group_indices = group.index.to_numpy()
    chunks = [chunk.tolist() for chunk in np.array_split(group_indices, N_SPLITS)]
    chunks.sort(key=len, reverse=True)

    for chunk in chunks:
        smallest_fold = min(range(N_SPLITS), key=lambda i: (len(fold_indices[i]), i))
        fold_indices[smallest_fold].extend(chunk)

parts = []
for fold_id, indices in enumerate(fold_indices, start=1):
    part_df = shuffled_df.loc[indices].sample(frac=1, random_state=RANDOM_STATE + fold_id).reset_index(drop=True)
    parts.append(part_df)

sizes = [len(part) for part in parts]
print("Fold sizes:", sizes)
print("Size difference:", max(sizes) - min(sizes))

Fold sizes: [50, 50, 49, 49, 49]
Size difference: 1


In [5]:
summary_rows = []
overall_dist = shuffled_df[STRATIFY_COL].value_counts(normalize=True).sort_index()

for fold_id, part_df in enumerate(parts, start=1):
    output_path = DATA_DIR / OUTPUT_TEMPLATE.format(part=fold_id)
    part_df.to_csv(output_path, index=False)

    label_counts = part_df[STRATIFY_COL].value_counts().sort_index()
    label_ratio = part_df[STRATIFY_COL].value_counts(normalize=True).sort_index()
    max_delta = label_ratio.reindex(overall_dist.index, fill_value=0).sub(overall_dist, fill_value=0).abs().max()

    summary_rows.append(
        {
            "part": fold_id,
            "rows": len(part_df),
            "output": output_path.name,
            "max_label_ratio_delta": round(float(max_delta), 4),
        }
    )

    print(f"Saved {output_path.name} with {len(part_df)} rows")
    display(label_counts.to_frame(name="count").T)

pd.DataFrame(summary_rows)

Saved test_part_1.csv with 50 rows


label,0,1,10,11,100,101,110,111,1000,1001,...,10110,10111,11000,11001,11010,11011,11100,11101,11110,11111
count,1,1,2,2,2,2,2,1,1,1,...,2,2,1,1,1,2,1,4,1,2


Saved test_part_2.csv with 50 rows


label,0,1,10,11,100,101,110,111,1001,1010,...,10110,10111,11000,11001,11010,11011,11100,11101,11110,11111
count,1,1,3,2,1,1,1,1,2,2,...,1,2,1,1,2,2,2,4,1,2


Saved test_part_3.csv with 49 rows


...

Saved test_part_4.csv with 49 rows


label,0,1,10,11,100,101,110,111,1000,1001,...,10110,10111,11000,11001,11010,11011,11100,11101,11110,11111
count,1,1,3,1,1,1,2,1,1,1,...,1,1,1,2,1,1,1,4,1,1


Saved test_part_5.csv with 49 rows


label,0,10,11,100,101,110,111,1000,1001,1010,...,10101,10110,10111,11001,11010,11011,11100,11101,11110,11111
count,2,3,1,1,2,2,1,1,1,2,...,2,2,1,2,1,1,1,3,1,2


,part,rows,output,max_label_ratio_delta
0,1,50,test_part_1.csv,0.0167
1,2,50,test_part_2.csv,0.0167
2,3,49,test_part_3.csv,0.0165
3,4,49,test_part_4.csv,0.0165
4,5,49,test_part_5.csv,0.0165
